# 02 · Cleaning

The cleaning rules live in `src/clean.py`. Every number they use (medians etc.) is learned from the training data only, then applied to train, validation and December.

## Fit the cleaner on training data and flag bad rates
A load is flagged when its rate per mile is below 0.6× or above 1.5× the median for the same equipment and distance band. The gap between normal and bad loads is wide, so the exact cut-offs don't matter much.

In [1]:
import sys; sys.path.append("../src")
from clean import Cleaner, fit_rate_baseline, outlier_mask
import pandas as pd

train_raw = pd.read_csv("../data/train_test.csv")
val_raw = pd.read_csv("../data/validation.csv")
dec_raw = pd.read_csv("../data/december_chart_inputs.csv")

cleaner = Cleaner().fit(train_raw)          # learns from train only
train = cleaner.transform(train_raw)
val = cleaner.transform(val_raw)
dec = cleaner.transform(dec_raw)

baseline = fit_rate_baseline(train)
train["is_outlier"] = outlier_mask(train, baseline)
print("flagged outliers:", train["is_outlier"].sum(), f"({train['is_outlier'].mean():.2%})")

flagged outliers: 677 (1.41%)


## Sanity checks
Row counts unchanged, no missing values left, and how often each flag fires.

In [2]:
for name, df, raw in [("train", train, train_raw), ("val", val, val_raw), ("dec", dec, dec_raw)]:
    assert len(df) == len(raw), f"{name}: row count changed"
    print(f"{name}: shape={df.shape}, NaN in weight={df['weight'].isna().sum()}, "
          f"min weight={df['weight'].min()}")
    if "market_index" in df:
        print("   NaN in market_index:", df["market_index"].isna().sum())

print(train[["weight_was_negative", "weight_capped", "weight_floored",
             "weight_missing", "market_index_missing", "is_outlier"]].mean().round(4))
print(val[["weight_was_negative", "weight_capped", "weight_floored",
           "weight_missing", "market_index_missing"]].mean().round(4))

train: shape=(48000, 20), NaN in weight=0, min weight=5000.0
   NaN in market_index: 0
val: shape=(12000, 18), NaN in weight=0, min weight=5000.0
   NaN in market_index: 0
dec: shape=(31, 11), NaN in weight=0, min weight=32000
weight_was_negative     0.0061
weight_capped           0.0251
weight_floored          0.0007
weight_missing          0.0062
market_index_missing    0.0078
is_outlier              0.0141
dtype: float64
weight_was_negative     0.0121
weight_capped           0.0251
weight_floored          0.0006
weight_missing          0.0138
market_index_missing    0.0208
dtype: float64
